In [ ]:
# ==============================================================
# CÓDIGO COMPLETO — 13 MODELOS
# 11 TRANSFER LEARNING + 2 CNN DO ZERO (lr=5e-4 e lr=1e-4)
#
# ✅ FUNCIONA COM SUA ESTRUTURA ATUAL DE PASTAS:
#    TREINAMENTO/{BOLSAS TREINAMENTO, CALÇAS TREINAMENTO}
#    TESTE/{BOLSAS TESTE, CALÇAS TESTE}
#
# Repeated Holdout (10 seeds) com métricas + tempos
#
# Salva POR MODELO:
#   - Excel (Rodadas + Resumo)
#   - LaTeX (Rodadas + Resumo) para Overleaf
#   - 3 gráficos separados em PDF
#
# Salva GERAL:
#   - Excel consolidado (todos os modelos)
#   - LaTeX consolidado (resumo geral + tabelas comparativas)
# ==============================================================

import os
import time
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ==============================================================
# 1) MONTAR DRIVE (GOOGLE COLAB)
# ==============================================================

from google.colab import drive
drive.mount('/content/drive')   # aviso "already mounted" pode ser ignorado

# ==============================================================
# 2) DIRETÓRIOS (AJUSTE AQUI SE PRECISAR)
# ==============================================================

BASE_DIR   = '/content/drive/MyDrive/Anna Tcc'
TREINO_DIR = os.path.join(BASE_DIR, 'TREINAMENTO')
TESTE_DIR  = os.path.join(BASE_DIR, 'TESTE')

# ==============================================================
# 3) CLASSES (CONFORME SUA ESTRUTURA ATUAL)
# ==============================================================

CLASSES_TREINO = ["BOLSAS TREINAMENTO", "CALÇAS TREINAMENTO"]
CLASSES_TESTE  = ["BOLSAS TESTE", "CALÇAS TESTE"]

# ==============================================================
# 4) SAÍDAS
# ==============================================================

OUT_DIR       = os.path.join(BASE_DIR, 'resultados_tabelas_13_modelos')
OUT_EXCEL_DIR = os.path.join(OUT_DIR, 'excel')
OUT_LATEX_DIR = os.path.join(OUT_DIR, 'latex')
OUT_PDF_DIR   = os.path.join(OUT_DIR, 'graficos_pdf')

os.makedirs(OUT_DIR,       exist_ok=True)
os.makedirs(OUT_EXCEL_DIR, exist_ok=True)
os.makedirs(OUT_LATEX_DIR, exist_ok=True)
os.makedirs(OUT_PDF_DIR,   exist_ok=True)

# ==============================================================
# 5) CHECAGEM (EVITA ERRO SILENCIOSO)
# ==============================================================

print("BASE_DIR existe? ", os.path.exists(BASE_DIR), BASE_DIR)
print("TREINO_DIR existe?", os.path.exists(TREINO_DIR), TREINO_DIR)
print("TESTE_DIR existe? ", os.path.exists(TESTE_DIR),  TESTE_DIR)

for c in CLASSES_TREINO:
    p = os.path.join(TREINO_DIR, c)
    print("Treino classe existe?", os.path.exists(p), p)

for c in CLASSES_TESTE:
    p = os.path.join(TESTE_DIR, c)
    print("Teste classe existe? ", os.path.exists(p), p)

assert os.path.exists(TREINO_DIR), f"Não encontrei: {TREINO_DIR}"
assert os.path.exists(TESTE_DIR),  f"Não encontrei: {TESTE_DIR}"

# ==============================================================
# 6) FUNÇÕES AUXILIARES
# ==============================================================

def ic95_t(media, desvio, n):
    """
    Intervalo de confiança de 95% para a média,
    usando distribuição t-Student para n <= 20.
    """
    if n <= 1 or np.isnan(desvio):
        return (np.nan, np.nan)

    t_table = {
        2: 12.706, 3: 4.303, 4: 3.182, 5: 2.776, 6: 2.571,
        7: 2.447, 8: 2.365, 9: 2.306, 10: 2.262,
        11: 2.228, 12: 2.201, 13: 2.179, 14: 2.160, 15: 2.145,
        16: 2.131, 17: 2.120, 18: 2.110, 19: 2.101, 20: 2.093
    }

    tcrit = t_table.get(n, 1.96)  # aproximação normal se n>20
    erro = tcrit * (desvio / math.sqrt(n))
    return (media - erro, media + erro)


def salvar_graficos_em_pdfs(df, model_name, out_dir=OUT_PDF_DIR):
    """
    Salva 3 gráficos em PDF:
      (1) Acurácia por rodada
      (2) Boxplot das métricas
      (3) Média ± desvio das métricas
    """
    os.makedirs(out_dir, exist_ok=True)

    # 1) Acurácia por rodada
    plt.figure(figsize=(7, 4))
    plt.plot(range(1, len(df) + 1), df['Acuracia'], marker='o')
    plt.title(f'Acurácia por Rodada — {model_name}')
    plt.xlabel('Rodada')
    plt.ylabel('Acurácia')
    plt.grid(alpha=0.3)
    out_pdf1 = os.path.join(out_dir, f"{model_name}_01_acuracia_por_rodada.pdf")
    plt.tight_layout()
    plt.savefig(out_pdf1, format="pdf")
    plt.close()

    # 2) Boxplot métricas
    plt.figure(figsize=(7, 4))
    df[['Acuracia', 'Precision', 'Recall', 'F1_score']].boxplot()
    plt.title(f'Distribuição das Métricas — {model_name}')
    plt.xticks(rotation=45)
    out_pdf2 = os.path.join(out_dir, f"{model_name}_02_boxplot_metricas.pdf")
    plt.tight_layout()
    plt.savefig(out_pdf2, format="pdf")
    plt.close()

    # 3) Média ± Desvio (métricas)
    plt.figure(figsize=(7, 4))
    metrics = ['Acuracia', 'Precision', 'Recall', 'F1_score']
    means   = df[metrics].mean()
    stds    = df[metrics].std()
    x = np.arange(len(metrics))
    plt.bar(x, means, yerr=stds, capsize=5)
    plt.xticks(x, ['Acc', 'Prec', 'Rec', 'F1'])
    plt.title(f'Performance Média ± Desvio — {model_name}')
    out_pdf3 = os.path.join(out_dir, f"{model_name}_03_media_mais_desvio.pdf")
    plt.tight_layout()
    plt.savefig(out_pdf3, format="pdf")
    plt.close()

    print("✅ PDFs salvos:")
    print("   ", out_pdf1)
    print("   ", out_pdf2)
    print("   ", out_pdf3)


def salvar_tabelas_excel_latex(df, resumo, model_name,
                               out_excel_dir=OUT_EXCEL_DIR,
                               out_latex_dir=OUT_LATEX_DIR):
    """
    Salva:
      - Excel com planilhas: Rodadas, Resumo
      - LaTeX com 2 tabelas: rodadas e resumo
    """
    os.makedirs(out_excel_dir, exist_ok=True)
    os.makedirs(out_latex_dir, exist_ok=True)

    # Excel
    out_xlsx = os.path.join(out_excel_dir, f"tabelas_{model_name}.xlsx")
    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="Rodadas", index=False)
        resumo.to_excel(writer, sheet_name="Resumo", index=True)

    # LaTeX – rodadas
    out_tex_rodadas = os.path.join(out_latex_dir, f"tabela_rodadas_{model_name}.tex")
    out_tex_resumo  = os.path.join(out_latex_dir, f"tabela_resumo_{model_name}.tex")

    df_latex = df.copy()
    for c in ["Acuracia", "Precision", "Recall", "F1_score"]:
        df_latex[c] = df_latex[c].map(lambda x: f"{x:.4f}")
    for c in ["Tempo_Treino_s", "Tempo_Pred_s", "Tempo_Total_s"]:
        df_latex[c] = df_latex[c].map(lambda x: f"{x:.2f}")
    df_latex.to_latex(out_tex_rodadas, index=False, escape=False)

    resumo_latex = resumo.copy()
    for c in resumo_latex.columns:
        resumo_latex[c] = resumo_latex[c].map(
            lambda x: f"{x:.4f}" if pd.notna(x) else ""
        )
    resumo_latex.to_latex(out_tex_resumo, index=True, escape=False)

    print("✅ Tabelas salvas:")
    print("   📌 Excel:", out_xlsx)
    print("   📌 LaTeX rodadas:", out_tex_rodadas)
    print("   📌 LaTeX resumo :", out_tex_resumo)

# ==============================================================
# 7) BUILDERS (Transfer Learning e CNN do zero)
# ==============================================================

def build_transfer_learning_model(base_model_fn, input_shape, lr=1e-4):
    """
    Constrói um modelo de Transfer Learning congelando o backbone
    e adicionando camadas densas no topo.
    """
    base_model = base_model_fn(input_shape=input_shape)
    base_model.trainable = False

    model = Sequential([
        base_model,
        GlobalAveragePooling2D(),
        Dense(128, activation='relu'),
        Dense(64, activation='relu'),
        Dense(2, activation='softmax')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


def build_cnn_from_scratch(input_shape, lr=5e-4):
    """
    Constrói uma CNN simples do zero.
    O learning rate será controlado externamente via parâmetro lr.
    """
    network = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        MaxPool2D(),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPool2D(),
        Conv2D(128, (3, 3), activation='relu'),
        MaxPool2D(),
        Flatten(),
        Dense(128, activation='relu'),
        Dense(64, activation='relu'),
        Dense(2, activation='softmax')
    ])

    network.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return network

# ==============================================================
# 8) PIPELINE ÚNICO (TL ou CNN)
# ==============================================================

def rodar_experimento_generico(
    MODEL_NAME,
    IMG_SIZE,
    preprocess_input_fn,
    model_builder_fn,
    epochs=30,
    batch_size=32,
    lr=1e-4,
    seeds=10,
    augment_params=None
):
    """
    Roda o experimento Repeated Holdout com 'seeds' diferentes.
    Suporta tanto modelos de Transfer Learning quanto CNN do zero.
    """
    resultados = []

    if augment_params is None:
        augment_params = dict(
            rotation_range=15,
            zoom_range=0.2,
            width_shift_range=0.1,
            height_shift_range=0.1,
            horizontal_flip=True
        )

    for rodada in range(seeds):
        print(f"\n🔁 {MODEL_NAME} | RODADA {rodada + 1}/{seeds} | SEED = {rodada}")

        SEED = rodada
        random.seed(SEED)
        np.random.seed(SEED)
        tf.random.set_seed(SEED)
        os.environ['PYTHONHASHSEED'] = str(SEED)
        os.environ['TF_DETERMINISTIC_OPS'] = '1'

        # Geradores
        if preprocess_input_fn is None:
            gerador_treinamento = ImageDataGenerator(rescale=1. / 255, **augment_params)
            gerador_teste       = ImageDataGenerator(rescale=1. / 255)
        else:
            gerador_treinamento = ImageDataGenerator(
                preprocessing_function=preprocess_input_fn,
                **augment_params
            )
            gerador_teste = ImageDataGenerator(
                preprocessing_function=preprocess_input_fn
            )

        dataset_treinamento = gerador_treinamento.flow_from_directory(
            TREINO_DIR,
            target_size=IMG_SIZE,
            batch_size=batch_size,
            class_mode='categorical',
            classes=CLASSES_TREINO,  # força as classes corretas
            shuffle=True,
            seed=SEED
        )

        dataset_teste = gerador_teste.flow_from_directory(
            TESTE_DIR,
            target_size=IMG_SIZE,
            batch_size=batch_size,
            class_mode='categorical',
            classes=CLASSES_TESTE,   # força as classes corretas
            shuffle=False
        )

        # Checagem de consistência
        if dataset_treinamento.num_classes != 2 or dataset_teste.num_classes != 2:
            raise ValueError("O código foi escrito para 2 classes. Ajuste as pastas/classes se necessário.")

        # Construir modelo
        model = model_builder_fn(
            input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
            lr=lr
        )

        early_stop = EarlyStopping(
            monitor='loss',
            patience=5,
            restore_best_weights=True,
            verbose=0
        )

        # Tempo de treino
        t0 = time.perf_counter()
        model.fit(dataset_treinamento, epochs=epochs, callbacks=[early_stop], verbose=0)
        t1 = time.perf_counter()
        tempo_treino = t1 - t0

        # Tempo de predição
        t2 = time.perf_counter()
        probs = model.predict(dataset_teste, verbose=0)
        t3 = time.perf_counter()
        tempo_pred = t3 - t2

        tempo_total = tempo_treino + tempo_pred

        # Métricas
        y_pred = np.argmax(probs, axis=1)
        y_true = dataset_teste.classes

        acc  = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
        rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)
        f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0)

        print(
            f"📊 Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f} | "
            f"Treino(s): {tempo_treino:.2f} | Pred(s): {tempo_pred:.2f} | Total(s): {tempo_total:.2f}"
        )

        resultados.append({
            "Modelo": MODEL_NAME,
            "Rodada": rodada + 1,
            "Seed":   SEED,
            "Acuracia":       acc,
            "Precision":      prec,
            "Recall":         rec,
            "F1_score":       f1,
            "Tempo_Treino_s": tempo_treino,
            "Tempo_Pred_s":   tempo_pred,
            "Tempo_Total_s":  tempo_total
        })

        tf.keras.backend.clear_session()

    # DataFrame com resultados por rodada
    df = pd.DataFrame(resultados)

    # Estatísticas resumo
    cols   = ["Acuracia", "Precision", "Recall", "F1_score",
              "Tempo_Treino_s", "Tempo_Pred_s", "Tempo_Total_s"]
    media  = df[cols].mean()
    desvio = df[cols].std()
    cv     = desvio / media

    n = len(df)
    ic_low, ic_high = [], []
    for c in cols:
        lo, hi = ic95_t(media[c], desvio[c], n)
        ic_low.append(lo)
        ic_high.append(hi)

    resumo = pd.DataFrame({
        "Media":         media,
        "Desvio_Padrao": desvio,
        "Coef_Variacao": cv,
        "IC95_Lower":    ic_low,
        "IC95_Upper":    ic_high
    })
    resumo.index.name = "Metrica"

    # Salvar saídas individuais
    salvar_tabelas_excel_latex(df, resumo, MODEL_NAME)
    salvar_graficos_em_pdfs(df, MODEL_NAME)

    return df, resumo

# ==============================================================
# 9) BACKBONES + PREPROCESS (TRANSFER LEARNING)
# ==============================================================

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as pp_mnv2

from tensorflow.keras.applications import EfficientNetB0, EfficientNetB1, EfficientNetB2
from tensorflow.keras.applications.efficientnet import preprocess_input as pp_eff

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet import preprocess_input as pp_resnet

from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input as pp_densenet

from tensorflow.keras.applications import MobileNetV3Small, MobileNetV3Large
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as pp_mnv3

from tensorflow.keras.applications import NASNetMobile
from tensorflow.keras.applications.nasnet import preprocess_input as pp_nasnet

from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input as pp_incv3

from tensorflow.keras.applications import Xception
from tensorflow.keras.applications.xception import preprocess_input as pp_xception

# ==============================================================
# 10) ESPECIFICAÇÕES DOS 13 MODELOS
#    11 BACKBONES DE TRANSFER LEARNING + 2 CNN DO ZERO
# ==============================================================

MODELOS = []

# Parâmetros padrão de aumento para TL
AUG_TL = dict(
    rotation_range=15,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

# --------------------------------------------------------------
# 10.1) 11 MODELOS DE TRANSFER LEARNING
#      (todos com lr = 1e-4)
# --------------------------------------------------------------

# 1) MobileNetV2
MODELOS.append({
    "MODEL_NAME": "mobilenetv2",
    "IMG_SIZE": (224, 224),
    "preprocess_input_fn": pp_mnv2,
    "model_builder_fn": lambda input_shape, lr: build_transfer_learning_model(
        base_model_fn=lambda input_shape: MobileNetV2(
            input_shape=input_shape, include_top=False, weights="imagenet"
        ),
        input_shape=input_shape,
        lr=lr
    ),
    "epochs": 30,
    "batch_size": 32,
    "lr": 1e-4,
    "augment_params": AUG_TL
})

# 2) EfficientNetB0
MODELOS.append({
    "MODEL_NAME": "efficientnetb0",
    "IMG_SIZE": (224, 224),
    "preprocess_input_fn": pp_eff,
    "model_builder_fn": lambda input_shape, lr: build_transfer_learning_model(
        base_model_fn=lambda input_shape: EfficientNetB0(
            input_shape=input_shape, include_top=False, weights="imagenet"
        ),
        input_shape=input_shape,
        lr=lr
    ),
    "epochs": 30,
    "batch_size": 32,
    "lr": 1e-4,
    "augment_params": AUG_TL
})

# 3) EfficientNetB1
MODELOS.append({
    "MODEL_NAME": "efficientnetb1",
    "IMG_SIZE": (240, 240),
    "preprocess_input_fn": pp_eff,
    "model_builder_fn": lambda input_shape, lr: build_transfer_learning_model(
        base_model_fn=lambda input_shape: EfficientNetB1(
            input_shape=input_shape, include_top=False, weights="imagenet"
        ),
        input_shape=input_shape,
        lr=lr
    ),
    "epochs": 30,
    "batch_size": 32,
    "lr": 1e-4,
    "augment_params": AUG_TL
})

# 4) EfficientNetB2
MODELOS.append({
    "MODEL_NAME": "efficientnetb2",
    "IMG_SIZE": (260, 260),
    "preprocess_input_fn": pp_eff,
    "model_builder_fn": lambda input_shape, lr: build_transfer_learning_model(
        base_model_fn=lambda input_shape: EfficientNetB2(
            input_shape=input_shape, include_top=False, weights="imagenet"
        ),
        input_shape=input_shape,
        lr=lr
    ),
    "epochs": 30,
    "batch_size": 32,
    "lr": 1e-4,
    "augment_params": AUG_TL
})

# 5) ResNet50
MODELOS.append({
    "MODEL_NAME": "resnet50",
    "IMG_SIZE": (224, 224),
    "preprocess_input_fn": pp_resnet,
    "model_builder_fn": lambda input_shape, lr: build_transfer_learning_model(
        base_model_fn=lambda input_shape: ResNet50(
            input_shape=input_shape, include_top=False, weights="imagenet"
        ),
        input_shape=input_shape,
        lr=lr
    ),
    "epochs": 30,
    "batch_size": 32,
    "lr": 1e-4,
    "augment_params": AUG_TL
})

# 6) DenseNet121
MODELOS.append({
    "MODEL_NAME": "densenet121",
    "IMG_SIZE": (224, 224),
    "preprocess_input_fn": pp_densenet,
    "model_builder_fn": lambda input_shape, lr: build_transfer_learning_model(
        base_model_fn=lambda input_shape: DenseNet121(
            input_shape=input_shape, include_top=False, weights="imagenet"
        ),
        input_shape=input_shape,
        lr=lr
    ),
    "epochs": 30,
    "batch_size": 32,
    "lr": 1e-4,
    "augment_params": AUG_TL
})

# 7) MobileNetV3Small
MODELOS.append({
    "MODEL_NAME": "mobilenetv3small",
    "IMG_SIZE": (224, 224),
    "preprocess_input_fn": pp_mnv3,
    "model_builder_fn": lambda input_shape, lr: build_transfer_learning_model(
        base_model_fn=lambda input_shape: MobileNetV3Small(
            input_shape=input_shape, include_top=False, weights="imagenet"
        ),
        input_shape=input_shape,
        lr=lr
    ),
    "epochs": 30,
    "batch_size": 32,
    "lr": 1e-4,
    "augment_params": AUG_TL
})

# 8) MobileNetV3Large
MODELOS.append({
    "MODEL_NAME": "mobilenetv3large",
    "IMG_SIZE": (224, 224),
    "preprocess_input_fn": pp_mnv3,
    "model_builder_fn": lambda input_shape, lr: build_transfer_learning_model(
        base_model_fn=lambda input_shape: MobileNetV3Large(
            input_shape=input_shape, include_top=False, weights="imagenet"
        ),
        input_shape=input_shape,
        lr=lr
    ),
    "epochs": 30,
    "batch_size": 32,
    "lr": 1e-4,
    "augment_params": AUG_TL
})

# 9) NASNetMobile
MODELOS.append({
    "MODEL_NAME": "nasnetmobile",
    "IMG_SIZE": (224, 224),
    "preprocess_input_fn": pp_nasnet,
    "model_builder_fn": lambda input_shape, lr: build_transfer_learning_model(
        base_model_fn=lambda input_shape: NASNetMobile(
            input_shape=input_shape, include_top=False, weights="imagenet"
        ),
        input_shape=input_shape,
        lr=lr
    ),
    "epochs": 30,
    "batch_size": 32,
    "lr": 1e-4,
    "augment_params": AUG_TL
})

# 10) InceptionV3
MODELOS.append({
    "MODEL_NAME": "inceptionv3",
    "IMG_SIZE": (299, 299),
    "preprocess_input_fn": pp_incv3,
    "model_builder_fn": lambda input_shape, lr: build_transfer_learning_model(
        base_model_fn=lambda input_shape: InceptionV3(
            input_shape=input_shape, include_top=False, weights="imagenet"
        ),
        input_shape=input_shape,
        lr=lr
    ),
    "epochs": 30,
    "batch_size": 32,
    "lr": 1e-4,
    "augment_params": AUG_TL
})

# 11) Xception
MODELOS.append({
    "MODEL_NAME": "xception",
    "IMG_SIZE": (299, 299),
    "preprocess_input_fn": pp_xception,
    "model_builder_fn": lambda input_shape, lr: build_transfer_learning_model(
        base_model_fn=lambda input_shape: Xception(
            input_shape=input_shape, include_top=False, weights="imagenet"
        ),
        input_shape=input_shape,
        lr=lr
    ),
    "epochs": 30,
    "batch_size": 32,
    "lr": 1e-4,
    "augment_params": AUG_TL
})

# --------------------------------------------------------------
# 10.2) 2 MODELOS CNN DO ZERO
#       - cnn_from_scratch_lr5e4
#       - cnn_from_scratch_lr1e4
# --------------------------------------------------------------

AUG_CNN = dict(
    rotation_range=15,
    zoom_range=0.25,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2]
)

# CNN do zero com lr = 5e-4
MODELOS.append({
    "MODEL_NAME": "cnn_from_scratch_lr5e4",
    "IMG_SIZE": (150, 150),
    "preprocess_input_fn": None,   # CNN pura usa rescale=1/255
    "model_builder_fn": lambda input_shape, lr: build_cnn_from_scratch(
        input_shape=input_shape, lr=lr
    ),
    "epochs": 50,
    "batch_size": 32,
    "lr": 5e-4,
    "augment_params": AUG_CNN
})

# CNN do zero com lr = 1e-4
MODELOS.append({
    "MODEL_NAME": "cnn_from_scratch_lr1e4",
    "IMG_SIZE": (150, 150),
    "preprocess_input_fn": None,   # CNN pura usa rescale=1/255
    "model_builder_fn": lambda input_shape, lr: build_cnn_from_scratch(
        input_shape=input_shape, lr=lr
    ),
    "epochs": 50,
    "batch_size": 32,
    "lr": 1e-4,
    "augment_params": AUG_CNN
})

# ==============================================================
# 11) EXECUTAR TODOS OS MODELOS E GERAR CONSOLIDADOS
# ==============================================================

resultados_gerais = []
resumos_gerais    = []

for cfg in MODELOS:
    print("\n" + "=" * 70)
    print(f"▶ Rodando modelo: {cfg['MODEL_NAME']}")
    print("=" * 70)

    df_modelo, resumo_modelo = rodar_experimento_generico(
        MODEL_NAME          = cfg["MODEL_NAME"],
        IMG_SIZE            = cfg["IMG_SIZE"],
        preprocess_input_fn = cfg["preprocess_input_fn"],
        model_builder_fn    = cfg["model_builder_fn"],
        epochs              = cfg["epochs"],
        batch_size          = cfg["batch_size"],
        lr                  = cfg["lr"],
        seeds               = 10,
        augment_params      = cfg["augment_params"]
    )

    # Garante que o nome do modelo conste nas tabelas de resumo mantidas em memória
    if "Modelo" not in df_modelo.columns:
        df_modelo.insert(0, "Modelo", cfg["MODEL_NAME"])
    if "Modelo" not in resumo_modelo.columns:
        resumo_modelo.insert(0, "Modelo", cfg["MODEL_NAME"])

    resultados_gerais.append(df_modelo)
    resumos_gerais.append(resumo_modelo)

# DataFrames gerais em memória (opcional para uso posterior)
df_todos     = pd.concat(resultados_gerais, ignore_index=True)
resumo_todos = pd.concat(resumos_gerais,    ignore_index=True)

# ==============================================================
# 12) FUNÇÃO PARA RECONSTRUIR CONSOLIDADOS A PARTIR DOS EXCELS
# ==============================================================

def reconstruir_consolidados():
    """
    Lê todos os arquivos 'tabelas_<modelo>.xlsx' em OUT_EXCEL_DIR
    e reconstrói:
      - Excel consolidado (tabelas_gerais_13_modelos.xlsx)
      - LaTeX resumo geral (tabela_resumo_13_modelos.tex)
      - LaTeX comparativo de médias (métricas e tempos)
    """
    assert os.path.exists(OUT_EXCEL_DIR), "OUT_EXCEL_DIR não existe."
    assert os.path.exists(OUT_LATEX_DIR), "OUT_LATEX_DIR não existe."

    arquivos = [
        f for f in os.listdir(OUT_EXCEL_DIR)
        if f.startswith("tabelas_") and f.endswith(".xlsx")
    ]

    if len(arquivos) == 0:
        raise FileNotFoundError(
            "Não encontrei arquivos 'tabelas_*.xlsx' em OUT_EXCEL_DIR."
        )

    tabelas_rodadas = []
    tabelas_resumo  = []

    for arq in sorted(arquivos):
        caminho      = os.path.join(OUT_EXCEL_DIR, arq)
        nome_modelo  = arq.replace("tabelas_", "").replace(".xlsx", "")

        df_rod = pd.read_excel(caminho, sheet_name="Rodadas")
        df_res = pd.read_excel(caminho, sheet_name="Resumo")

        # Padroniza: garante presença do campo Modelo
        if "Modelo" not in df_rod.columns:
            df_rod.insert(0, "Modelo", nome_modelo)
        if "Modelo" not in df_res.columns:
            df_res.insert(0, "Modelo", nome_modelo)

        tabelas_rodadas.append(df_rod)
        tabelas_resumo.append(df_res)

    df_todos = pd.concat(tabelas_rodadas, ignore_index=True)
    resumo_todos = pd.concat(tabelas_resumo, ignore_index=True)

    # ===== Excel consolidado
    out_geral_xlsx = os.path.join(OUT_EXCEL_DIR, "tabelas_gerais_13_modelos.xlsx")
    with pd.ExcelWriter(out_geral_xlsx, engine="openpyxl") as writer:
        df_todos.to_excel(writer, sheet_name="Rodadas_Todos", index=False)
        resumo_todos.to_excel(writer, sheet_name="Resumo_Todos", index=False)

    print("✅ Excel geral atualizado em:", out_geral_xlsx)

    # ===== LaTeX consolidado (resumo geral)
    out_tex_resumo_geral = os.path.join(OUT_LATEX_DIR, "tabela_resumo_13_modelos.tex")
    resumo_todos_latex = resumo_todos.copy()

    for c in ["Media", "Desvio_Padrao", "Coef_Variacao", "IC95_Lower", "IC95_Upper"]:
        if c in resumo_todos_latex.columns:
            resumo_todos_latex[c] = resumo_todos_latex[c].map(
                lambda x: f"{x:.4f}" if pd.notna(x) else ""
            )

    resumo_todos_latex.to_latex(out_tex_resumo_geral, index=False, escape=False)
    print("✅ LaTeX (resumo geral) atualizado em:", out_tex_resumo_geral)

    # ===== tabelas comparativas (médias)
    if ("Metrica" in resumo_todos.columns) and ("Media" in resumo_todos.columns):
        metrica_media = resumo_todos.query(
            "Metrica in ['Acuracia','Precision','Recall','F1_score']"
        ).pivot(index="Modelo", columns="Metrica", values="Media")

        tempo_media = resumo_todos.query(
            "Metrica in ['Tempo_Treino_s','Tempo_Pred_s','Tempo_Total_s']"
        ).pivot(index="Modelo", columns="Metrica", values="Media")

        out_tex_media_metricas = os.path.join(
            OUT_LATEX_DIR, "tabela_medias_metricas_por_modelo_13.tex"
        )
        out_tex_media_tempos = os.path.join(
            OUT_LATEX_DIR, "tabela_medias_tempos_por_modelo_13.tex"
        )

        metrica_media.to_latex(out_tex_media_metricas, float_format="%.4f")
        tempo_media.to_latex(out_tex_media_tempos,   float_format="%.4f")

        print("✅ LaTeX (médias métricas) atualizado em:", out_tex_media_metricas)
        print("✅ LaTeX (médias tempos) atualizado em:", out_tex_media_tempos)
    else:
        print("⚠️ Não foi possível gerar tabelas comparativas "
              "(faltou 'Metrica'/'Media' no resumo).")

# Chamada final para consolidar tudo após rodar os 13 modelos
reconstruir_consolidados()

print("\n✅ Pronto. Os 13 modelos foram executados e os consolidados foram gerados/atualizados.")
